# About cross spectrum, coherence, and phase estimation

The goal of this "tutorial" is to write a "physicist-like" summary (main results and key things to remember, no proofs or heavy derivations) for myself, about cross spectra, spectral coherence, and cross-spectral phase. This should typically be read after the similar tutorials about [gaussian stochastic process spectra](https://github.com/jerabaul29/tutorials/blob/main/uncertainty_gaussian_spectrum/uncertainty_gaussian_spectrum.ipynb) and [bicoherence](https://github.com/jerabaul29/tutorials/blob/main/Bicoherence/Bicoherence.ipynb) . As a note, it may have been more logical to look at cross spectrum etc (the present tutorial) before bicoherence, but it just happened I wrote these down in the opposite order :) .

If having more questions, I recommend chatting with an AI - AIs have read a lot about this and with proper user check and critical thinking are very useful. I would have struggled to write this tutorial without AI - the results are typically scattered, hard to find, and either not available online at all or paywalled.

The key references for this tutorial (according to AI):

Goodman, N. R. (1957). "On the joint estimation of the spectra, cospectrum and quadrature spectrum of a two-dimensional stationary Gaussian process." Scientific Paper No. 10, Engineering Statistics Laboratory, New York University. 

Carter, G. C., Knapp, C. H., & Nuttall, A. H. (1973). "Estimation of the magnitude-squared coherence function via overlapped fast Fourier transform processing." IEEE Transactions on Audio and Electroacoustics.

Koopmans, L. H. (1995). The Spectral Analysis of Time Series. Academic Press. 

Nuttall, A. H. (1971). "Spectral estimation by means of overlapped FFT processing of windowed data." Naval Underwater Systems Center Report.

Bendat, J. S. and Piersol, A. G. (2000) "Random data analysis and measurement procedures." Measurement Science and Technology.
Carter, G. C. (1987). "Coherence and time delay estimation." Proceedings of the IEEE, 75(2), 236-255.

## A bit of context about cross spectrum, coherence, and phase estimation for stochastic gaussian processes

In this tutorial, we consider stochastic gaussian processes. Typically, the underlying process is unknown, and can only be observed through some individual realization of the process, typically a timeseries (for example, a waves in ice vertical acceleration time series as I will use in the following in my work [arXiv:2507.19034](https://arxiv.org/pdf/2507.19034) ; but any other kind of data following a similar gaussian process will be described by the same laws).

In this tutorial, we focus on the cross spectral density and the coherence and phase estimation that come out of it.

Formally and "abstractly", stochastic gaussian processes are defined from their underlying (usually not observable and hidden) self and cross correlation functions. In the case of interest here (cross correlation), the formal definition of the cross spectral density is as the Fourier transform of the cross-correlation function $R_{XY}(\tau)$ between two stochastic gaussian processes $X$ and $Y$:

$S_{XY}(f) = \int_{-\infty}^{\infty} R_{XY} (\tau) e^{-j 2 \pi f \tau} d \tau$,

where $R_{XY}(\tau) = E[X(t) Y(t + \tau)]$, $\tau$ the time lag.

In practical, discretized terms, the canonical estimator for $S_{XY}$ is:

$P_{xy}(f) = \left\langle FFT_x^*(f) FFT_y(f) \right\rangle$,

where $\left\langle . \right\rangle$ means "averaging over segments", $x(t)$ and $y(t)$ are the synchronized discrete real world realizations of $X$ and $Y$, and $FFT_x$ and $FFT_y$ their relative discrete fourier transforms (DFTs). Note that $P_{xy}$ is an array of complex values and has all the usual 2-sided vs 1-sided and conventions caveats that are usual of DFTs (see the tutorial about the power spectrum of gaussian stochastic processes linked above).

Given this, one can define the coherence as:

$C_{xy}(f) = \frac{|P_{xy}(f)|^2}{P_{xx}(f) P_{yy}(f)}$ .

Observe that modulus is taken (these are generally complex values), and that the order of operations is important. In particular for $|P_{xy}(f)|^2$, the average over segments is taken first, then the modulus square is taken: spelled out otherwise:

$C_{xy}(f) = \frac{|\left\langle P_{xy}(f) \right\rangle |^2}{\left\langle P_{xx} \right\rangle \left\langle P_{yy} \right\rangle}$ .

Note that $C_{xy}(f)$ as defined here is the *magnitude-squared coherence* (MSC): it is the estimator of the squared true coherence $\gamma^2(f)$ (with $\gamma(f)$ the, unsquared, true underlying coherence introduced further below), not of $\gamma(f)$ itself. This distinction matters because $\gamma$ (not squared) is the quantity that appears in the distributions given later in this tutorial.

The consequence is that at a given frequency $f$, if the relative phase between the signals $x$ and $y$ is fully locked across the segments over which the averaging is done, the coherence will be equal to 1 at this frequency; if the signals are not phase locked one with another, the coherence estimator will not be exactly 0, but will be biased slightly above 0 (see the bias of the null distribution given below) and should be compared against a significance threshold rather than against exactly 0. This is the same idea as for bicoherence (except bicoherence looked at the coupling between 3 different frequencies, while here we only look at 2).

Naturally, to reduce noise, the usual tricks from the tutorial discussing Welch periodograms applies, i.e. tapering functions, windowing and overlap.

In cases where the coherence is significantly non-zero, it is meaningful to look at the cross-spectrum phase estimator:

$\theta_{xy}(f) = \arg{(P_{xy}(f))}$,

where $\arg$ is the argument or "phase" operator.

Therefore, the coherence and cross-spectrum phase are useful to check if two signals $x$ and $y$ are significantly "phase-locked" over time at a given frequency, and if so what their relative phases are.

## Putting error bars on the estimators

As for the previous tutorials, $C_{xy}(f)$ and $\theta_{xy}(f)$ are only estimators for the hidden underlying true value. For stochastic gaussian processes, one can actually compute (either exactly analytically, or asymptotically) the distribution / mean / bias / standard deviation of these estimators relatively to the true underlying value.

### Case with underlying coherence truly equal to zero

In this case, $C_{xy}$ follows a beta distribution with parameters $\alpha=1$ and $\beta = N_{eff}-1$, with $N_{eff}$ the effective number of independent segments. This means that the cumulative density function (CDF) simplifies as:

$P(C_{xy} <= c) = 1 - (1-c)^{N_{eff}-1}$

The mean (equal to the bias) is $E[C_{xy}] = Bias{(C_{xy})} = 1/N_{eff}$, and the standard deviation (note the distribution is a beta distribution which is skewed, not for example a gaussian) is $\sigma = \frac{1}{N_{eff}} \sqrt{\frac{N_{eff}-1}{N_{eff}+1}} \approx 1 / N_{eff}$ . This means that one needs to be significantly above the bias to ensure that there is true non-zero coherence.

In practice, this null CDF gives a direct way to build a significance threshold: for a chosen confidence level (e.g. 95%), the coherence value $c_{crit}$ that a purely-noise (zero true coherence) signal pair would exceed only 5% of the time is obtained by solving $P(C_{xy} <= c_{crit}) = 0.95$, i.e.:

$c_{crit} = 1 - (1-0.95)^{1/(N_{eff}-1)} = 1 - 0.05^{1/(N_{eff}-1)}$ .

Any estimated coherence value above $c_{crit}$ can then be considered significantly non-zero at the chosen confidence level.

In this case, the phase $\theta_{xy}(f)$ is uniformly distributed on the whole phase range (typically depending on the arg function used $[-\pi; \pi[$).

### Case with underlying coherence truly strictly positive (non zero)

When the true underlying coherence is strictly positive, the distributions change. Given an underlying true coherence $\gamma(f)$, the estimator $C_{xy}(f)$ follows a Goodman distribution (transformed hypergeometric function distribution) with probability density function (PDF) $f$:

$f(C_{xy}(f)) = (N_{eff}-1)(1-\gamma(f)^2)^{N_{eff}}(1-C_{xy}(f))^{N_{eff}-2} {}_2F_1(N_{eff},N_{eff};1;\gamma(f)^2 C_{xy}(f))$ ,

where $_2F_1$ is the Gauss hypergeometric function. In real world uses, $\gamma(f)$ is unknown and the best one can do is replace it with $C_{xy}(f)$.

The mean is approximately:

$E(C_{xy}(f)) = \gamma^2 + \frac{1}{N_{eff}}(1-\gamma^2)^2$, which means the bias is approximately $Bias(C_{xy}(f)) \approx \frac{1}{N_{eff}}(1-\gamma^2)^2$ and has strictly speaking to be compensated for,

and the standard deviation is approximately $\sigma(C_{xy}(f)) \approx \frac{\sqrt{2} \gamma (1-\gamma^2)}{\sqrt{N_{eff}}}$ .

The phase-error PDF and its large-$N_{eff}$ standard deviation given below are from Carter (1987); the 1973 Carter/Knapp/Nuttall paper only covers the MSC statistics, not the phase.

The phase estimator is unbiased and has a PDF for the phase error $\Delta \theta (f) = \theta_{xy}(f) - \theta_0(f)$ with $\theta_0(f)$ the true underlying phase:

$f(\Delta \theta (f)) \approx \frac{1}{2 \pi} (1 - \gamma^2(f))^{N_{eff}} {}_2F_1(N_{eff}, 1; 1/2; \gamma^2(f) \cos^2(\Delta \theta (f)))$ ,

which converges to a gaussian distribution for large $N_{eff}$.

This is unbiased ($E(\theta_{xy}(f)) = \theta_0(f)$), and has a standard deviation that converges for large enough $N_{eff}$ to:

$\sigma_{\theta}(f) \approx \sqrt{\frac{1-\gamma^2(f)}{2 N_{eff} \gamma^2(f)}}$ (in rads).

So the higher $\gamma$ (estimated by $C_{xy}$) and the effective number of segments $N_{eff}$, the more accurate the phase estimate.

A practical consequence is that, once $\gamma$ is significantly non-zero, one can also build a confidence interval on the estimated phase itself, in the usual gaussian large-$N_{eff}$ approximation: $\theta_0(f) \in [\theta_{xy}(f) - z \sigma_{\theta}(f); \theta_{xy}(f) + z \sigma_{\theta}(f)]$ at confidence level corresponding to $z$ (e.g. $z \approx 1.96$ for 95%), analogously to the significance threshold $c_{crit}$ built above for the coherence.

The effective number of segments is computed based on the windowing function and the overlap similarly as for the Welch estimator:

$N_{eff} = \frac{N}{1 + 2 \sum_{m=1}^{N-1} (1 - \frac{m}{N}) \rho^2(m \times S)}$,

where:

- $N$ is the total number of overlapping segments,
- $\rho(m \times S)$ is the normalized correlation coefficient between 2 segments shifted by $m$ "welch segments" steps (i.e. shifted by $m \times S$ sample points, where $S$ is how many points separate the start of one segment to the start of the next segment: $S=nperseg - noverlap$, with $nperseg$ the number of points per segment and $noverlap$ the number of overlapping points between 2 consecutive segments), which is computed from the windowing function $w$ as:
$\rho(m \times S) = \frac{\sum_n w[n] w[n+m \times S]}{\sum_n w^2[n]}$ .

Note that all the Goodman/Carter distributions and approximations above (the non-null coherence PDF, its mean/bias/std, and the phase-error PDF/std) are derived assuming $N_{eff}$ independent segments. In practice, with overlapped Welch segments, the segments are not truly independent; $N_{eff}$ as computed here is only an approximate correction accounting for this residual correlation, so the resulting significance thresholds and error bars should themselves be treated as approximate, especially for small $N_{eff}$ or heavy overlap.

## Example: looking at the wave propagation between 2 points on sea ice

These are example data taken from the paper [arXiv:2507.19034](https://arxiv.org/pdf/2507.19034) . We only consider a small segment of the data considered there.

In [ ]:
# TODO

## Bonus: Monte Carlo check of the estimator distributions and uncertainties

This is to illustrate the results from above with Monte Carlo simulations. These take the typical coherence and phase shift from the example above, and do stochastic rollouts to compare empirical vs. the above theoretical estimators distributions.

In [ ]:
#TODO